from datasets import load_dataset
import pandas as pd, re, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load dataset
ds = load_dataset("emotion")
train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

# Clean text
def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", "", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()

train['text_clean'] = train['text'].apply(clean_text)
val['text_clean'] = val['text'].apply(clean_text)
test['text_clean'] = test['text'].apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train['text_clean'])
X_test = vectorizer.transform(test['text_clean'])
y_train, y_test = train['label'], test['label']

# Train Logistic Regression
clf = LogisticRegression(max_iter=200, solver='saga', multi_class='multinomial')
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=ds['train'].features['label'].names))


In [ ]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
import pandas as pd, re, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load dataset
ds = load_dataset("emotion")
train = ds['train'].to_pandas()
val = ds['validation'].to_pandas()
test = ds['test'].to_pandas()

# Clean text
def clean_text(s):
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", "", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()

train['text_clean'] = train['text'].apply(clean_text)
val['text_clean'] = val['text'].apply(clean_text)
test['text_clean'] = test['text'].apply(clean_text)

# TF-IDF
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(train['text_clean'])
X_test = vectorizer.transform(test['text_clean'])
y_train, y_test = train['label'], test['label']

# Train Logistic Regression
clf = LogisticRegression(max_iter=200, solver='saga', multi_class='multinomial')
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=ds['train'].features['label'].names))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.8385
              precision    recall  f1-score   support

     sadness       0.87      0.91      0.89       581
         joy       0.79      0.96      0.87       695
        love       0.83      0.52      0.64       159
       anger       0.90      0.76      0.82       275
        fear       0.87      0.74      0.80       224
    surprise       0.88      0.35      0.50        66

    accuracy                           0.84      2000
   macro avg       0.86      0.71      0.75      2000
weighted avg       0.84      0.84      0.83      2000



In [ ]:
import numpy as np
def predict_emotion(text):
    x = vectorizer.transform([clean_text(text)])
    pred = clf.predict(x)[0]
    label = ds['train'].features['label'].names[pred]
    print(f"Text: {text}\nPredicted emotion: {label}")

# Try it
predict_emotion("I am feeling very happy today!")
predict_emotion("I am really sad and depressed.")
predict_emotion("I am so angry with what happened.")


Text: I am feeling very happy today!
Predicted emotion: joy
Text: I am really sad and depressed.
Predicted emotion: sadness
Text: I am so angry with what happened.
Predicted emotion: anger


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np evaluate

# Load data
dataset = load_dataset("emotion")
num_labels = dataset['train'].features['label'].num_classes

# Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids","attention_mask","labels"])

# Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels),
        "f1": f1.compute(predictions=preds, references=labels, average="macro")
    }

# Training args
training_args = TrainingArguments(
    output_dir="./bert-emotion",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
!pip install numpy pandas scikit-learn matplotlib seaborn nltk textblob datasets transformers torch sentencepiece joblib evaluate


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

# Load data
dataset = load_dataset("emotion")
num_labels = dataset['train'].features['label'].num_classes

# Tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids","attention_mask","labels"])

# Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels),
        "f1": f1.compute(predictions=preds, references=labels, average="macro")
    }

# Training args
training_args = TrainingArguments(
    output_dir="./bert-emotion",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [2]:
# Upgrade transformers, datasets, evaluate and related libs
!pip install -q --upgrade transformers datasets evaluate accelerate huggingface_hub sentencepiece

# (optional) ensure torch is present and reasonably up-to-date
!pip install -q --upgrade torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 10.9 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(mod

In [ ]:
import os
os._exit(0)


In [4]:
import transformers, torch, evaluate
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Evaluate:", evaluate.__version__)
if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")


Transformers: 5.0.0
Torch: 2.10.0+cu128
Evaluate: 0.4.6
GPU detected: Tesla T4


In [ ]:
!pip install -q evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.2 MB/s eta 0:00:00


In [ ]:
!pip install -q transformers datasets evaluate torch sentencepiece


In [3]:
import transformers, torch, evaluate
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Evaluate:", evaluate.__version__)
if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")


Transformers: 5.0.0
Torch: 2.10.0+cu128
Evaluate: 0.4.6
GPU detected: Tesla T4


In [ ]:
!pip install -q numpy pandas scikit-learn matplotlib seaborn nltk textblob datasets transformers torch sentencepiece joblib evaluate


In [5]:
import transformers, torch, evaluate

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("Evaluate:", evaluate.__version__)

if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected — go to Runtime → Change runtime type → select T4 GPU → Save")


Transformers: 5.0.0
Torch: 2.10.0+cu128
Evaluate: 0.4.6
GPU detected: Tesla T4


In [ ]:
!pip install -q evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
import numpy as np, evaluate


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

# Load dataset
dataset = load_dataset("emotion")
num_labels = dataset['train'].features['label'].num_classes

# Tokenizer setup
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids","attention_mask","labels"])

# Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels),
        "f1": f1.compute(predictions=preds, references=labels, average="macro")
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./bert-emotion",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    load_best_model_at_end=True
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
!pip install -U transformers


In [ ]:
!pip install -q --upgrade transformers datasets evaluate accelerate huggingface_hub
!pip install -q --upgrade torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 707.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.1 MB/s eta 

In [ ]:
!pip uninstall -y cudf-cu12 pylibcudf-cu12 fastai torchvision torchaudio pyarrow


Found existing installation: cudf-cu12 25.6.0
Uninstalling cudf-cu12-25.6.0:
  Successfully uninstalled cudf-cu12-25.6.0
Found existing installation: pylibcudf-cu12 25.6.0
Uninstalling pylibcudf-cu12-25.6.0:
  Successfully uninstalled pylibcudf-cu12-25.6.0
Found existing installation: fastai 2.8.4
Uninstalling fastai-2.8.4:
  Successfully uninstalled fastai-2.8.4
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Found existing installation: pyarrow 18.1.0
Uninstalling pyarrow-18.1.0:
  Successfully uninstalled pyarrow-18.1.0


In [ ]:
!pip install -U torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -U transformers datasets evaluate


Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torchvision==0.23.0 (from versions: 0.1.6, 0.2.0, 0.17.0+cu121, 0.17.1+cu121, 0.17.2+cu121, 0.18.0+cu121, 0.18.1+cu121, 0.19.0+cu121, 0.19.1+cu121, 0.20.0+cu121, 0.20.1+cu121)
ERROR: No matching distribution found for torchvision==0.23.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch, transformers, datasets, evaluate

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)

print("GPU Available:", torch.cuda.is_available())


Torch: 2.8.0+cu126
Transformers: 4.57.1
Datasets: 4.3.0
Evaluate: 0.4.6
GPU Available: True


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load the dataset (emotion dataset from HuggingFace)
dataset = load_dataset("dair-ai/emotion")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

tokenized_datasets = dataset.map(tokenize, batched=True)

# Load model (6 emotion labels)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"]}

training_args = TrainingArguments(
    output_dir="./bert-emotion",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-730071594.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.213400,0.177547,0.926000
2,0.088100,0.141690,0.935000


In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Load the fine-tuned model
model_path = "./bert-emotion"
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Function to predict emotion
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_label = torch.argmax(logits, dim=1).item()

    # Emotion labels based on the dataset mapping (modify if needed)
    labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

    return labels[predicted_label]

# TEST
user_input = "I am so happy today!"
print("Emotion:", predict_emotion(user_input))


Emotion: joy


In [ ]:
!pip install gradio


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import gradio as gr

# ✅ Load trained model (your model folder)
model_path = "./bert-emotion"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

# ✅ Emotion labels (same order as dataset used)
emotion_labels = ["sadness", "joy", "love", "anger", "fear", "surprise"]

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities).item()

    return {
        "Emotion": emotion_labels[prediction],
        "Confidence Score": float(probabilities[0][prediction])
    }

interface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(label="Enter text here"),
    outputs="json",
    title="Emotion Detection using BERT",
    description="Type any sentence. The model predicts the emotional tone.",
)

interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://643f54583927d2dc85.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from huggingface_hub import login
login()


In [ ]:
!huggingface-cli whoami


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
KEERTHANA6805


In [ ]:
from huggingface_hub import HfApi, upload_folder

api = HfApi()

repo_id = "KEERTHANA6805/emotion-detection-bert"  # you can change the name if you want

upload_folder(
    folder_path="bert-emotion",
    repo_id=repo_id,
    repo_type="model",
)

print("Model uploaded to Hugging Face!")
print("Link:", f"https://huggingface.co/{repo_id}")


In [ ]:
from huggingface_hub import login
login()


In [ ]:
import os

token = ""  # paste your token inside quotes: "hf_abc...xyz"
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_TOKEN"] = token

!git config --global credential.helper store
!git config --global user.email "your-email@example.com"
!git config --global user.name "KEERTHANA6805"


In [ ]:
!huggingface-cli whoami


⚠️  Warning: 'huggingface-cli whoami' is deprecated. Use 'hf auth whoami' instead.
KEERTHANA6805


In [ ]:
# ✅ Save model into a folder named "model"
model.save_pretrained("model")
tokenizer.save_pretrained("model")


NameError: name 'model' is not defined

In [10]:
from transformers import BertForSequenceClassification, BertTokenizer

# Load the trained model from the folder where Trainer saved it
model = BertForSequenceClassification.from_pretrained("./bert-emotion")
tokenizer = BertTokenizer.from_pretrained("./bert-emotion")

ImportError: cannot import name 'merge_with_config_defaults' from 'transformers.utils.generic' (/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py)

In [ ]:
!ls ./bert-emotion


In [ ]:
!ls -l ./bert-emotion


total 0


In [11]:
trainer.train()


NameError: name 'trainer' is not defined

In [ ]:
from datasets import load_dataset

dataset = load_dataset("emotion")


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6   # Emotion dataset has 6 labels
)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-emotion",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
!pip install --upgrade transformers


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert-emotion",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"]
)


In [ ]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: keerthana-2301135 (keerthana-2301135-sri-ramakrishna-engineering-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.251000,0.181431


Epoch,Training Loss,Validation Loss
1,0.251000,0.181431
2,0.127500,0.145260


TrainOutput(global_step=2000, training_loss=0.3091143522262573, metrics={'train_runtime': 3133.411, 'train_samples_per_second': 10.213, 'train_steps_per_second': 0.638, 'total_flos': 8419856154624000.0, 'train_loss': 0.3091143522262573, 'epoch': 2.0})

In [ ]:
model.save_pretrained("model")
tokenizer.save_pretrained("model")


('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/vocab.txt',
 'model/added_tokens.json')

In [ ]:
output_dir="./bert-emotion"


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained("./bert-emotion")


In [ ]:

model.save_pretrained("model")
tokenizer.save_pretrained("model")


('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/vocab.txt',
 'model/added_tokens.json')

In [ ]:
!ls model


config.json	   special_tokens_map.json  vocab.txt
model.safetensors  tokenizer_config.json


In [ ]:
trainer.push_to_hub()


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...emotion/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...715841.9901ffca169f.577.0:   2%|1         |   235B / 14.6kB            

  ...emotion/training_args.bin:   2%|1         |  93.0B / 5.84kB            

CommitInfo(commit_url='https://huggingface.co/KEERTHANA6805/bert-emotion/commit/3ccf456c8678adeacde65627eb34d47ac5a222b5', commit_message='End of training', commit_description='', oid='3ccf456c8678adeacde65627eb34d47ac5a222b5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KEERTHANA6805/bert-emotion', endpoint='https://huggingface.co', repo_type='model', repo_id='KEERTHANA6805/bert-emotion'), pr_revision=None, pr_num=None)

In [ ]:
%%writefile model/README.md
# BERT Emotion Classifier

This model classifies text into 6 emotion categories:
- joy
- sadness
- anger
- fear
- love
- surprise

Model: `bert-base-uncased`
Dataset: `emotion` (Hugging Face datasets library)

### Usage (example code)

```python
from transformers import pipeline
classifier = pipeline("text-classification", model="KEERTHANA6805/bert-emotion")
classifier("I am so happy today!")


Overwriting model/README.md


---

### 📌 2. Create `LICENSE`

Run this in another code cell:

In [ ]:
%%writefile model/LICENSE
MIT License

Copyright (c) 2025 <KEERTHANA>

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.

Overwriting model/LICENSE


In [ ]:
!ls -l model


total 427956
-rw-r--r-- 1 root root       949 Oct 29 06:28 config.json
-rw-r--r-- 1 root root      1068 Oct 29 07:05 LICENSE
-rw-r--r-- 1 root root 437970952 Oct 29 06:28 model.safetensors
-rw-r--r-- 1 root root       398 Oct 29 07:03 README.md
-rw-r--r-- 1 root root       695 Oct 29 06:28 special_tokens_map.json
-rw-r--r-- 1 root root      1272 Oct 29 06:28 tokenizer_config.json
-rw-r--r-- 1 root root    231508 Oct 29 06:28 vocab.txt


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="model",
    repo_id="KEERTHANA6805/bert-emotion",
    commit_message="Upload final model"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9662: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t/model/model.safetensors:   2%|1         | 8.38MB /  438MB            

CommitInfo(commit_url='https://huggingface.co/KEERTHANA6805/bert-emotion/commit/5e61013fce05cdce9af926524b12600c2075f375', commit_message='Upload final model', commit_description='', oid='5e61013fce05cdce9af926524b12600c2075f375', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KEERTHANA6805/bert-emotion', endpoint='https://huggingface.co', repo_type='model', repo_id='KEERTHANA6805/bert-emotion'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="KEERTHANA6805/bert-emotion")
classifier("I am extremely happy today!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'LABEL_1', 'score': 0.9974572062492371}]

In [ ]:
from google.colab import files
files.download('/content/bert-emotion/pytorch_model.bin')
files.download('/content/bert-emotion/config.json')
files.download('/content/bert-emotion/tokenizer.json')
files.download('/content/bert-emotion/tokenizer_config.json')


FileNotFoundError: Cannot find file: /content/model/model.safetensors

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
/content/bert-emotion/
/content/outputs/
/content/drive/MyDrive/bert-emotion/


ls: cannot access '/content/bert-emotion/': No such file or directory
ls: cannot access '/content/outputs/': No such file or directory
ls: cannot access '/content/drive/MyDrive/bert-emotion/': No such file or directory


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.save_pretrained("/content/bert-emotion/")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

('/content/bert-emotion/tokenizer_config.json',
 '/content/bert-emotion/special_tokens_map.json',
 '/content/bert-emotion/vocab.txt',
 '/content/bert-emotion/added_tokens.json',
 '/content/bert-emotion/tokenizer.json')

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_folder(
    folder_path="/content/bert-emotion",
    repo_id="KEERTHANA6805/bert-emotion",
    commit_message="upload tokenizer files"
)


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6901e5da-2bb395121479302923634f7a;1d12092b-2c18-4876-a262-680ef1649d82)

Repository Not Found for url: https://huggingface.co/api/models/KEERTHANA6805/bert-emotion/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
from huggingface_hub import login
login()


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/bert-emotion",
    repo_id="KEERTHANA6805/bert-emotion",
    commit_message="Uploading tokenizer + model files"
)


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6901e656-5c6066b62382634240ad8ade;a5c686d6-dd52-48b7-8ba0-f5b9b0b90ff2)

Repository Not Found for url: https://huggingface.co/api/models/KEERTHANA6805/bert-emotion/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [ ]:
from huggingface_hub import login
login()


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/bert-emotion",
    repo_id="KEERTHANA6805/bert-emotion",   # MUST MATCH NAME EXACTLY
    commit_message="Uploading tokenizer + model files"
)


CommitInfo(commit_url='https://huggingface.co/KEERTHANA6805/bert-emotion/commit/0377fbcacef55c098c57f60c727cf24466b49a5a', commit_message='Uploading tokenizer + model files', commit_description='', oid='0377fbcacef55c098c57f60c727cf24466b49a5a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KEERTHANA6805/bert-emotion', endpoint='https://huggingface.co', repo_type='model', repo_id='KEERTHANA6805/bert-emotion'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/model",  # Use the correct path to your trained model
    repo_id="KEERTHANA6805/bert-emotion",
    commit_message="Uploading model weights"
)

ValueError: Provided path: '/content/model' is not a directory

In [ ]:
!find /content -name "pytorch_model.bin"


In [ ]:
!find /content -name "config.json"


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/bert-emotion",
    repo_id="KEERTHANA6805/bert-emotion",
    commit_message="Uploading model weights"
)


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/KEERTHANA6805/bert-emotion/commit/0377fbcacef55c098c57f60c727cf24466b49a5a', commit_message='Uploading model weights', commit_description='', oid='0377fbcacef55c098c57f60c727cf24466b49a5a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/KEERTHANA6805/bert-emotion', endpoint='https://huggingface.co', repo_type='model', repo_id='KEERTHANA6805/bert-emotion'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("/content/bert-emotion")
model.save_pretrained("/content/bert-emotion", safe_serialization=False)


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /content/bert-emotion.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="KEERTHANA6805/bert-emotion",
    local_dir="/content/bert-emotion",
    ignore_patterns=["*.msgpack", "*.h5"]
)


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/607 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

...

training_args.bin:   0%|          | 0.00/5.84k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

'/content/bert-emotion'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive
